In [ ]:
# ── CELL 1: GPU + deps ────────────────────────────────────────────────────────
!nvidia-smi
!pip install pytorch-metric-learning -q
import torch
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


In [ ]:
# ── CELL 2: Mount Drive + checkpoint dir ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/results/checkpoints', exist_ok=True)
print('Drive mounted.')


In [ ]:
# ── CELL 3: Dataset download (skip if present) ───────────────────────────────
import os, glob, zipfile, subprocess

DATA_ROOT = None
candidates = glob.glob('/content/data/**/PartAnnotation', recursive=True) + \
             glob.glob('/content/drive/MyDrive/**/PartAnnotation', recursive=True)
if candidates:
    DATA_ROOT = os.path.dirname(candidates[0])
    print(f'Dataset found: {DATA_ROOT}')
else:
    print('Downloading ShapeNet Part...')
    subprocess.run(['wget', '-q', '-O', '/content/shapenet_part.zip',
        'https://shapenet.cs.stanford.edu/ericyi/shapenetcore_partanno_segmentation_benchmark_v0.zip'],
        check=True)
    subprocess.run(['unzip', '-q', '/content/shapenet_part.zip', '-d', '/content/data/'], check=True)
    candidates = glob.glob('/content/data/**/PartAnnotation', recursive=True)
    DATA_ROOT = os.path.dirname(candidates[0])
    print(f'Downloaded to: {DATA_ROOT}')


In [ ]:
# ── CELL 4: Dataset loader ────────────────────────────────────────────────────
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

SYNSET_TO_CLASS = {
    '02691156':0,'02773838':1,'02954340':2,'02958343':3,'03001627':4,
    '03261776':5,'03467517':6,'03624134':7,'03636649':8,'03642806':9,
    '03790512':10,'03797390':11,'03948459':12,'04099429':13,'04225987':14,'04379243':15
}

def load_pts(path):
    pts = []
    with open(path) as f:
        for line in f:
            v = line.strip().split()
            if len(v) >= 3: pts.append([float(x) for x in v[:3]])
    return np.array(pts, dtype='float32')

def load_seg(path):
    segs = []
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s: segs.append(int(s))
    return np.array(segs, dtype='int64')

class ShapeNet_coseg(Dataset):
    def __init__(self, partition='train', num_points=1024, obj_class=4,
                 data_root=None, train_ratio=0.8):
        self.num_points = num_points
        if data_root is None: data_root = DATA_ROOT
        target_syn = next((s for s,c in SYNSET_TO_CLASS.items() if c==obj_class), None)
        syn_dir = os.path.join(data_root, target_syn)
        pts_dir = os.path.join(syn_dir, 'points')
        seg_dir = os.path.join(syn_dir, 'points_label')
        all_ids = sorted([f[:-4] for f in os.listdir(pts_dir) if f.endswith('.pts')])
        seg_map = {}
        for dirpath, _, files in os.walk(seg_dir):
            for f in files:
                if f.endswith('.seg') and f[:-4] not in seg_map:
                    seg_map[f[:-4]] = os.path.join(dirpath, f)
        valid_ids = [i for i in all_ids if i in seg_map]
        np.random.seed(42)
        perm  = np.random.permutation(len(valid_ids))
        split = int(len(valid_ids) * train_ratio)
        chosen = [valid_ids[i] for i in (perm[:split] if partition=='train' else perm[split:])]
        self.samples = [(os.path.join(pts_dir,sid+'.pts'), seg_map[sid]) for sid in chosen]
        print(f'[ShapeNet] {partition}: {len(self.samples)} samples  (class={obj_class})')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        pc  = load_pts(self.samples[idx][0])
        seg = load_seg(self.samples[idx][1])
        n   = min(len(pc), len(seg))
        pc, seg = pc[:n], seg[:n]
        N   = len(pc)
        idx_s = np.random.choice(N, self.num_points, replace=(N < self.num_points))
        pc, seg = pc[idx_s], seg[idx_s]
        pc -= pc.mean(0); s = np.max(np.linalg.norm(pc, axis=1))
        if s > 0: pc /= s
        binary = (seg > seg.min()).astype('int64')
        return pc.astype('float32'), binary

print('Dataset loader defined.')


In [ ]:

# ────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

# ── DGCNN helpers (unchanged) ──────────────────────────────────────────────────
def knn_graph(x, k):
    xt = x.permute(0,2,1)
    return torch.cdist(xt,xt).topk(k+1,dim=-1,largest=False).indices[:,:,1:]

def get_edge_feature(x, idx):
    B,D,N = x.shape; k = idx.shape[2]
    xt   = x.permute(0,2,1).contiguous()
    flat = idx.reshape(B,-1)
    nb   = torch.gather(xt,1,flat.unsqueeze(-1).expand(B,N*k,D)).view(B,N,k,D)
    xi   = xt.unsqueeze(2).expand(B,N,k,D)
    return torch.cat([xi, nb-xi], dim=-1).permute(0,3,1,2)

class EdgeConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=20):
        super().__init__()
        self.k   = k
        self.net = nn.Sequential(
            nn.Conv2d(in_ch*2, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2))
    def forward(self, x):
        return self.net(get_edge_feature(x, knn_graph(x, self.k))).max(dim=-1)[0]

class DGCNN(nn.Module):
    def __init__(self, k=20, emb_dim=512):
        super().__init__()
        self.ec1 = EdgeConv(3,   64,  k)
        self.ec2 = EdgeConv(64,  64,  k)
        self.ec3 = EdgeConv(64,  128, k)
        self.ec4 = EdgeConv(128, 256, k)
        self.proj = nn.Sequential(
            nn.Conv1d(512, emb_dim, 1, bias=False),
            nn.BatchNorm1d(emb_dim), nn.LeakyReLU(0.2), nn.Dropout(0.4))
    def forward(self, x):
        f1=self.ec1(x); f2=self.ec2(f1); f3=self.ec3(f2); f4=self.ec4(f3)
        return self.proj(torch.cat([f1,f2,f3,f4],dim=1))

# ── Mutual Attention Module ────────────────────────────────────────────────────
class MutualAttentionModule(nn.Module):
    
    def __init__(self, in_dim, att_dim=128):
        super().__init__()
        # Q, K, V projections  (applied as 1D convolutions over points)
        self.Wq = nn.Conv1d(in_dim, att_dim, 1, bias=False)
        self.Wk = nn.Conv1d(in_dim, att_dim, 1, bias=False)
        self.Wv = nn.Conv1d(in_dim, att_dim, 1, bias=False)
        # Residual projection back to in_dim
        self.Wo = nn.Conv1d(att_dim, in_dim, 1, bias=False)
        self.bn = nn.BatchNorm1d(in_dim)
        self.scale = att_dim ** -0.5

    def forward(self, feats, mode='fg'):
        """
        feats : (B, D, N)
        mode  : 'fg' or 'bg'
        """
        B, D, N = feats.shape

        # Project to Q, K, V:   each (B, att_dim, N)
        Q = self.Wq(feats)   # (B, C', N)
        K = self.Wk(feats)   # (B, C', N)
        V = self.Wv(feats)   # (B, C', N)

        # ── Cross-cloud attention ──────────────────────────────────────────────
        # For each sample i (anchor), average over all other samples j (reference)
        # attention(i,j) = softmax(Q_i @ K_j^T)   shape (N, N)
        # col_avg(i,j)   = mean over rows  → shape (N,) — highlights ref points
        #                                              with strong correspondence
        # We accumulate col_avg across all references and average.

        enhanced_sum = torch.zeros_like(V)   # accumulator (B, C', N)
        n_refs = 0

        for j in range(B):
            for i in range(B):
                if i == j:
                    continue
                # Q_j: (C', N_j) → (N_j, C')  anchor queries
                # K_i: (C', N_i) → (N_i, C')  reference keys
                Qj = Q[j].permute(1, 0)          # (N, C')
                Ki = K[i].permute(1, 0)          # (N, C')
                Vi = V[i].permute(1, 0)          # (N, C')

                # Pairwise correlation: (N_j, N_i)
                scores = torch.matmul(Qj, Ki.t()) * self.scale   # (N, N)

                # Row-wise softmax: each anchor point softly assigned to ref points
                row_soft = torch.softmax(scores, dim=-1)          # (N, N)

                # Column-wise mean: highlight reference points with strong mutual corr
                col_avg = row_soft.mean(dim=0)                    # (N,)  ref side

                if mode == 'fg':
                    att_map = col_avg                             # high-corr = fg
                else:
                    att_map = 1.0 - col_avg                      # complement = bg

                # Apply attention map to reference values: (N, C') * (N, 1) → (N, C')
                attended = att_map.unsqueeze(-1) * Vi             # (N, C')

                # Accumulate into anchor j's slot (we want enhanced features for j)
                # Note: we're building enhanced features for j using reference i
                # But what we actually need is the enhanced feature of the CURRENT cloud i
                # (the anchor in Figure 3 is i, reference is j)
                # Let's fix: anchor = i, reference = j
                Qi = Q[i].permute(1, 0)          # (N, C') anchor
                Kj = K[j].permute(1, 0)          # (N, C') reference
                Vj = V[j].permute(1, 0)          # (N, C')

                sc = torch.matmul(Qi, Kj.t()) * self.scale       # (N, N)
                rs = torch.softmax(sc, dim=-1)                    # (N, N)
                ca = rs.mean(dim=0)                               # (N,) ref side

                if mode == 'fg':
                    am = ca
                else:
                    am = 1.0 - ca

                att_ref = am.unsqueeze(-1) * Vj                   # (N, C')
                enhanced_sum[i] += att_ref.permute(1, 0)          # (C', N)
                n_refs += 1

        if n_refs > 0:
            # Average over all reference contributions
            # Each cloud i was referenced by (B-1) others, but we added B*(B-1) total
            # Correct: per cloud i, sum over j≠i → divide by (B-1)
            enhanced_avg = enhanced_sum / max(B - 1, 1)
        else:
            enhanced_avg = torch.zeros_like(V)

        # Residual connection: original features + attended features
        delta = self.bn(self.Wo(enhanced_avg))    # (B, D, N)
        return feats + delta


# ── MAS-equipped PointSampler ─────────────────────────────────────────────────
class MASampler(nn.Module):
    """
    PointSampler upgraded with Mutual Attention Sampling.

    Before scoring points for selection, the sampler first runs
    the MutualAttentionModule to enhance features with cross-cloud
    context, then scores the enhanced features.

    mode='fg' → emphasise high cross-cloud correlation (foreground)
    mode='bg' → emphasise low cross-cloud correlation  (background)
    """
    def __init__(self, in_dim, n_sample, att_dim=128, mode='fg'):
        super().__init__()
        self.n    = n_sample
        self.mode = mode
        self.mas  = MutualAttentionModule(in_dim, att_dim)
        self.net  = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(),
            nn.Linear(256, 128),    nn.ReLU(),
            nn.Linear(128, 1))

    def forward(self, feats):
        """
        feats : (B, D, N)
        Returns: sampled_feats (B, D, n_sample), indices, weights
        """
        B, D, N = feats.shape
        # Step 1: mutual attention enhancement
        enhanced = self.mas(feats, mode=self.mode)        # (B, D, N)
        # Step 2: score enhanced features
        w   = torch.softmax(
                  self.net(enhanced.permute(0,2,1)).squeeze(-1),
                  dim=-1)                                  # (B, N)
        idx = w.topk(self.n, dim=-1).indices               # (B, n_sample)
        # Step 3: gather from ORIGINAL feats (not enhanced) — paper design
        sampled = torch.gather(feats, 2,
                      idx.unsqueeze(1).expand(B, D, self.n))  # (B, D, n_sample)
        return sampled, idx, w


# ── Unchanged components ──────────────────────────────────────────────────────
class PartHead(nn.Module):
    def __init__(self, in_dim, n_parts=2):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_parts)
    def forward(self, feats):
        return torch.softmax(self.fc(feats.permute(0,2,1)), dim=-1)


# ── CoSegNet with MAS ─────────────────────────────────────────────────────────
class CoSegNet(nn.Module):
    """
    Full model:
      encoder  : DGCNN  (unchanged)
      fg_sampler: MASampler(mode='fg') — cross-cloud fg attention
      bg_sampler: MASampler(mode='bg') — cross-cloud bg attention
      part_head : 2-class soft assignment (unchanged)
    """
    def __init__(self, n_fg=256, n_bg=256, k=20, emb_dim=512,
                 n_parts=2, att_dim=128, use_mas=True):
        super().__init__()
        self.encoder = DGCNN(k=k, emb_dim=emb_dim)
        self.use_mas = use_mas
        if use_mas:
            self.fg_sampler = MASampler(emb_dim, n_fg, att_dim=att_dim, mode='fg')
            self.bg_sampler = MASampler(emb_dim, n_bg, att_dim=att_dim, mode='bg')
        else:
            # Fallback: original simple PointSampler (for ablation)
            from torch import nn as _nn
            class _PS(_nn.Module):
                def __init__(s, d, n):
                    super().__init__()
                    s.n = n
                    s.net = _nn.Sequential(
                        _nn.Linear(d,256),_nn.ReLU(),
                        _nn.Linear(256,128),_nn.ReLU(),
                        _nn.Linear(128,1))
                def forward(s, feats):
                    B,D,N = feats.shape
                    w   = torch.softmax(s.net(feats.permute(0,2,1)).squeeze(-1),dim=-1)
                    idx = w.topk(s.n,dim=-1).indices
                    return torch.gather(feats,2,idx.unsqueeze(1).expand(B,D,s.n)),idx,w
            self.fg_sampler = _PS(emb_dim, n_fg)
            self.bg_sampler = _PS(emb_dim, n_bg)
        self.part_head = PartHead(emb_dim, n_parts)

    def forward(self, xyz):
        feats = self.encoder(xyz.permute(0,2,1))         # (B, D, N)
        fg_f, fg_idx, fg_w = self.fg_sampler(feats)
        bg_f, bg_idx, bg_w = self.bg_sampler(feats)
        probs = self.part_head(feats)                    # (B, N, 2)
        return feats, fg_f, bg_f, fg_w, bg_w, probs

print('MAS-equipped CoSegNet defined.')
print('  MutualAttentionModule: cross-cloud Q/K/V attention')
print('  MASampler(fg): foreground-emphasised selection')
print('  MASampler(bg): background-emphasised selection')
print('  CoSegNet(use_mas=True/False): toggle for ablation')


In [ ]:
# ── CELL 6: Loss functions (all from previous sessions, unchanged) ────────────
from pytorch_metric_learning.losses import NTXentLoss
ntxent = NTXentLoss(temperature=0.07)

def contrastive_loss(fg_f, bg_f):
    B,D,_ = fg_f.shape
    fg_obj = fg_f.mean(dim=2); bg_obj = bg_f.mean(dim=2)
    emb    = torch.cat([fg_obj, bg_obj], dim=0)
    labels = torch.cat([torch.arange(B), torch.arange(B)]).to(fg_f.device)
    try:    return ntxent(emb, labels)
    except: return torch.tensor(0.0, device=fg_f.device)

def repulsion_loss(fg_f, bg_f):
    return F.cosine_similarity(fg_f.mean(2), bg_f.mean(2), dim=-1).mean()

def spatial_loss_uniform(feats, xyz, k=10):
    B,D,N = feats.shape
    knn_idx = torch.cdist(xyz,xyz).topk(k+1,dim=-1,largest=False).indices[:,:,1:]
    ft   = feats.permute(0,2,1).contiguous()
    flat = knn_idx.reshape(B,-1)
    nb   = torch.gather(ft,1,flat.unsqueeze(-1).expand(B,N*k,D)).view(B,N,k,D)
    fi   = ft.unsqueeze(2).expand(B,N,k,D)
    return ((fi-nb)**2).sum(-1).mean()

def entropy_loss(probs):
    return -(probs * torch.log(probs + 1e-8)).sum(-1).mean()

def ema_consistency_loss(student_probs, teacher_probs):
    """KL divergence: forces student to match teacher everywhere."""
    teacher_probs = teacher_probs.detach()
    eps = 1e-8
    kl = teacher_probs * (torch.log(teacher_probs + eps) - torch.log(student_probs + eps))
    return kl.sum(-1).mean()

print('All loss functions defined.')


In [ ]:
# ── CELL 7: EMA utilities (unchanged) ────────────────────────────────────────
import copy

def build_teacher(student):
    teacher = copy.deepcopy(student)
    for param in teacher.parameters():
        param.requires_grad = False
    return teacher

@torch.no_grad()
def ema_update(teacher, student, alpha=0.999):
    for t_p, s_p in zip(teacher.parameters(), student.parameters()):
        t_p.data.mul_(alpha).add_(s_p.data, alpha=(1.0 - alpha))

print('EMA utilities defined.')


In [ ]:
# ── CELL 8: Config + evaluation function ─────────────────────────────────────
import os
import numpy as np
from sklearn.metrics import jaccard_score, f1_score

CFG = {
    'obj_class':  4,      # Chair
    'num_points': 1024,
    'batch_size': 8,
    'n_fg':       256,
    'n_bg':       256,
    'n_parts':    2,
    'emb_dim':    512,
    'dgcnn_k':    20,
    'att_dim':    128,    # MAS attention dimension C' (paper uses 128)
}

def evaluate(model, test_loader):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xyz, lbl in test_loader:
            xyz = xyz.to(DEVICE)
            lbl = lbl.long()
            feats, fg_f, bg_f, _, _, _ = model(xyz)
            ft   = feats.permute(0,2,1)
            fg_p = fg_f.mean(2, keepdim=True).permute(0,2,1)
            bg_p = bg_f.mean(2, keepdim=True).permute(0,2,1)
            pred = ((ft-fg_p).norm(dim=-1) < (ft-bg_p).norm(dim=-1)).long()
            preds_all.append(pred.cpu().numpy().reshape(-1))
            labels_all.append(lbl.numpy().reshape(-1))
    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(labels_all)
    iou = jaccard_score(y_true, y_pred, average='binary', zero_division=0)
    f1  = f1_score(     y_true, y_pred, average='binary', zero_division=0)
    return iou, f1

print('Config and evaluate() defined.')


In [ ]:
# ── CELL 9: Training function ─────────────────────────────────────────────────
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

def run_experiment(
    tag              = 'exp',
    n_epochs         = 30,
    lr               = 3e-4,
    lambda_sp        = 0.01,
    k_sp             = 10,
    lambda_ent       = 0.0001,
    ema_alpha        = 0.999,
    lambda_ema       = 0.1,
    ema_warmup_epochs= 5,
    use_mas          = True,    # ← NEW: toggle MAS on/off
    att_dim          = 128,     # ← NEW: MAS attention dim
):
    mas_str = 'MAS' if use_mas else 'no-MAS'
    print(f'\n{"="*65}')
    print(f'  {tag.upper()}')
    print(f'  [{mas_str}] α={ema_alpha} | λ_ema={lambda_ema} | warmup={ema_warmup_epochs}ep')
    print(f'  λ_sp={lambda_sp} k={k_sp} | λ_ent={lambda_ent}')
    print(f'{"="*65}')

    train_ds = ShapeNet_coseg('train', CFG['num_points'], CFG['obj_class'])
    test_ds  = ShapeNet_coseg('test',  CFG['num_points'], CFG['obj_class'])
    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'],
                              shuffle=True, drop_last=True, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'],
                              shuffle=False, num_workers=2)

    student = CoSegNet(
        n_fg=CFG['n_fg'], n_bg=CFG['n_bg'], k=CFG['dgcnn_k'],
        emb_dim=CFG['emb_dim'], n_parts=CFG['n_parts'],
        att_dim=att_dim, use_mas=use_mas).to(DEVICE)
    teacher = build_teacher(student)

    print(f'Student params: {sum(p.numel() for p in student.parameters()):,}')

    opt   = optim.Adam(student.parameters(), lr=lr, weight_decay=1e-4)
    sched = StepLR(opt, step_size=15, gamma=0.5)

    best_iou, best_f1 = 0.0, 0.0
    best_ckpt = f'/content/results/checkpoints/{tag}_teacher_best.pt'
    history   = {'loss':[], 'iou':[], 'f1':[], 'iou_student':[]}

    for epoch in range(n_epochs):
        student.train(); teacher.eval()
        ep_loss = 0.0
        use_ema_this_epoch = (epoch >= ema_warmup_epochs)

        for xyz, _ in train_loader:
            xyz = xyz.to(DEVICE)

            # Student forward
            s_feats, s_fg_f, s_bg_f, _, _, s_probs = student(xyz)
            # Teacher forward (no grad)
            with torch.no_grad():
                _, _, _, _, _, t_probs = teacher(xyz)

            # Loss
            loss = (contrastive_loss(s_fg_f, s_bg_f)
                    + 0.5 * repulsion_loss(s_fg_f, s_bg_f)
                    + lambda_sp  * spatial_loss_uniform(s_feats, xyz, k_sp)
                    + lambda_ent * entropy_loss(s_probs))

            if use_ema_this_epoch:
                loss = loss + lambda_ema * ema_consistency_loss(s_probs, t_probs)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            opt.step()
            ema_update(teacher, student, alpha=ema_alpha)
            ep_loss += loss.item()

        sched.step()

        iou_t, f1_t = evaluate(teacher, test_loader)
        iou_s, _    = evaluate(student, test_loader)
        avg_loss    = ep_loss / len(train_loader)

        history['loss'].append(avg_loss)
        history['iou'].append(iou_t)
        history['f1'].append(f1_t)
        history['iou_student'].append(iou_s)

        marker = ''
        if iou_t > best_iou:
            best_iou, best_f1 = iou_t, f1_t
            torch.save(teacher.state_dict(), best_ckpt)
            marker = '  --> Best'

        warmup_str = ' [warmup]' if not use_ema_this_epoch else ''
        print(f'[{tag}] E{epoch:02d} | loss {avg_loss:.4f} | '
              f'T-IOU {iou_t:.4f} | S-IOU {iou_s:.4f} | '
              f'F1 {f1_t:.4f}{marker}{warmup_str}')

    print(f'DONE [{tag}]  Best Teacher IOU={best_iou:.4f}  F1={best_f1:.4f}')
    return history, best_iou, best_f1

print('run_experiment() defined.')


In [ ]:
# ── CELL 10: EMA baseline (no MAS) — control ─────────────────────────────────
h_base, iou_base, f1_base = run_experiment(
    tag        = 'ema_noMAS',
    n_epochs   = 30,
    use_mas    = False,
    ema_alpha  = 0.999,
    lambda_ema = 0.1,
    lambda_sp  = 0.01,
    lambda_ent = 0.0001,
)


In [ ]:
# ── CELL 11: MAS + EMA — main experiment ────────────────────────────────────
h_mas, iou_mas, f1_mas = run_experiment(
    tag        = 'ema_MAS',
    n_epochs   = 30,
    use_mas    = True,
    att_dim    = 128,
    ema_alpha  = 0.999,
    lambda_ema = 0.1,
    lambda_sp  = 0.01,
    lambda_ent = 0.0001,
)


In [ ]:
# ── CELL 12: MAS att_dim ablation ───────────────────────────────────────────
att_dims = [64, 128, 256]
ablation_results = {}

for ad in att_dims:
    h, iou, f1 = run_experiment(
        tag        = f'mas_attdim_{ad}',
        n_epochs   = 15,
        use_mas    = True,
        att_dim    = ad,
        ema_alpha  = 0.999,
        lambda_ema = 0.1,
        lambda_sp  = 0.01,
        lambda_ent = 0.0001,
    )
    ablation_results[ad] = {'iou': iou, 'f1': f1}

print('\n' + '='*55)
print(f'  att_dim  ablation results (15 epochs each)')
print('='*55)
print(f'  att_dim     IOU      F1')
print('-'*55)
for ad, r in ablation_results.items():
    best_marker = ' ← best' if r['iou'] == max(v['iou'] for v in ablation_results.values()) else ''
    print(f'  {ad:>7}   {r["iou"]:.4f}   {r["f1"]:.4f}{best_marker}')
print('='*55)


In [ ]:
# ── CELL 13: Comparison plots ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('MAS vs No-MAS: EMA Teacher Performance', fontsize=14)

# IOU over epochs
ax = axes[0]
ax.plot(h_base['iou'],   label='No-MAS (baseline)', color='steelblue',  linewidth=2)
ax.plot(h_mas['iou'],    label='With MAS',           color='darkorange', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Teacher IOU')
ax.set_title('IOU over Training'); ax.legend(); ax.grid(alpha=0.3)

# F1 over epochs
ax = axes[1]
ax.plot(h_base['f1'],    label='No-MAS (baseline)', color='steelblue',  linewidth=2)
ax.plot(h_mas['f1'],     label='With MAS',           color='darkorange', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Teacher F1')
ax.set_title('F1 over Training'); ax.legend(); ax.grid(alpha=0.3)

# att_dim bar chart
ax = axes[2]
dims = list(ablation_results.keys())
ious = [ablation_results[d]['iou'] for d in dims]
bars = ax.bar([str(d) for d in dims], ious, color=['#5B9BD5','#ED7D31','#70AD47'])
ax.set_xlabel('att_dim'); ax.set_ylabel('Best IOU (15 ep)')
ax.set_title('MAS Attention Dim Ablation'); ax.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, ious):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('/content/results/mas_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/results/mas_comparison.png')


In [ ]:
# ── CELL 14: Complete summary ────────────────────────────────────────────────
print('\n' + '='*70)
print('  COMPLETE RESULTS — MAS INTEGRATION')
print('='*70)

# Historical results from previous sessions
hist = {
    'Baseline (contrastive only)':     (0.3379, 0.5052),
    'Spatial + Entropy (15 ep)':       (0.4885, 0.6906),
    'EMA Teacher-Student (prev)':      (0.6086, 0.7567),
    'Conf-guided EMA thresh=0.8':      (0.6791, 0.8089),
}

print(f'  {"-"*68}')
print(f'  {"Method":<42}  {"IOU":>8}  {"F1":>8}')
print(f'  {"-"*68}')
for name, (iou, f1) in hist.items():
    print(f'  {name:<42}  {iou:>8.4f}  {f1:>8.4f}')
print(f'  {"-"*68}')
print(f'  {"EMA (no MAS) — this session":<42}  {iou_base:>8.4f}  {f1_base:>8.4f}')
print(f'  {"EMA + MAS — this session":<42}  {iou_mas:>8.4f}  {f1_mas:>8.4f}')

delta = iou_mas - iou_base
sign  = '+' if delta >= 0 else ''
print(f'  {"-"*68}')
print(f'  MAS effect: {sign}{delta:.4f} IOU  ({sign}{delta*100:.2f}%)')
print('='*70)

print('\nAblation (att_dim, 15 epochs):')
print(f'  {"att_dim":<12} {"IOU":>8} {"F1":>8}')
for ad, r in ablation_results.items():
    print(f'  {ad:<12} {r["iou"]:>8.4f} {r["f1"]:>8.4f}')


In [ ]:
# ── CELL 15: Save to Drive ───────────────────────────────────────────────────
import shutil, os

dest = '/content/drive/MyDrive/BTP_MAS_EMA'
os.makedirs(dest, exist_ok=True)

# Copy checkpoints
ckpt_src = '/content/results/checkpoints'
for f in os.listdir(ckpt_src):
    shutil.copy(os.path.join(ckpt_src, f), os.path.join(dest, f))

# Copy plot
if os.path.exists('/content/results/mas_comparison.png'):
    shutil.copy('/content/results/mas_comparison.png', dest)

# Save summary text
with open(os.path.join(dest, 'summary.txt'), 'w') as f:
    f.write('MAS + EMA Results\n')
    f.write(f'No-MAS  IOU={iou_base:.4f}  F1={f1_base:.4f}\n')
    f.write(f'With MAS IOU={iou_mas:.4f}  F1={f1_mas:.4f}\n')
    f.write(f'Delta: {iou_mas-iou_base:+.4f}\n')
    f.write('\nAblation:\n')
    for ad, r in ablation_results.items():
        f.write(f'  att_dim={ad}: IOU={r["iou"]:.4f}  F1={r["f1"]:.4f}\n')

print(f'Saved to Google Drive: {dest}')
print('Files saved:')
for f in os.listdir(dest):
    print(f'  {f}')
